In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))
    

class Router(nn.Module):
    def __init__(self, d_model, num_experts, top_k):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.top_k = top_k

    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)
        top_k_probs, top_k_indices = torch.topk(probs, self.top_k, dim=-1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        return top_k_probs, top_k_indices
    

class SparseMoE(nn.Module):
    def __init__(self, d_model, hidden_dim, num_experts=8, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        self.experts = nn.ModuleList([
            Expert(d_model, hidden_dim) for _ in range(num_experts)
        ])

        self.router = Router(d_model, num_experts, top_k)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model)

        routing_weights, selected_experts = self.router(x_flat)

        final_output = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.experts):
            expert_mask = (selected_experts == i)
            token_indices, k_indices = torch.where(expert_mask)

            if token_indices.numel() > 0:
                expert_inputs = x_flat[token_indices]

                expert_outputs = expert(expert_inputs)

                weights = routing_weights[token_indices, k_indices].unsqueeze(-1)

                final_output[token_indices] += expert_outputs * weights


